# RSVQA-HR pilot run (Colab)

Run this in your **Colab Pro** account with a **GPU runtime** (Runtime → Change runtime type → GPU).

Before the first run:
- `rsvqa_hr_zips` (shared from the 400 GB account) must be in this account's **My Drive** as a shortcut:
  Drive → Shared with me → right-click `rsvqa_hr_zips` → Organize → Add shortcut → My Drive.

Checkpoints, logs and result tables are written to `MyDrive/rsvqa_experiments/` as training runs,
so they survive a disconnect.

## 1. Mount Drive and check the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the latest code

In [ ]:
import os
import subprocess

REPO = '/content/rsvqa-hybrid-vqa'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/ngtdat9/rsvqa-hybrid-vqa.git', REPO], check=True)
%cd {REPO}
!git pull -q
!pip install -q -r requirements.txt

## 3. Get the HR data onto this machine

Extracts `Images.tar` straight from Drive (no 13.5 GB copy first). Takes several minutes.
Skipped if this session already has the images.

In [ ]:
from pathlib import Path

SRC = Path('/content/drive/MyDrive/rsvqa_hr_zips')
RAW = Path('data/raw/rsvqa_hr')
assert SRC.exists(), ('rsvqa_hr_zips is not in My Drive: share it from the 400 GB account '
                      'and add a shortcut to My Drive (see the top of this notebook)')
RAW.mkdir(parents=True, exist_ok=True)
if not (RAW / 'Data').exists():
    subprocess.run(f'cp {SRC}/*.json {RAW}/', shell=True, check=True)
    subprocess.run(['tar', '-xf', str(SRC / 'Images.tar'), '-C', str(RAW)], check=True)
print(len(list((RAW / 'Data').glob('*.png'))), 'images (expected 10659)')

## 4. Check the area ranges

Look at the `== Area answers in train` section. If a range shows **0.0%**, training will stop with a
`vocab_size` error: change `area_buckets` in `configs/hr_config.yaml` first.

In [ ]:
!python scripts/inspect_rsvqa.py data/raw/rsvqa_hr USGS_split

## 5. Train

Default: the pilot (1,500 train tiles, `configs/hr_config.yaml`) with the baseline settings.

- **Week 3 tuning:** set `RUN` to a name in `TUNING`; that run uses the baseline config plus those overrides.
- **Resume after a disconnect:** run steps 1-3, then set `RESUME` to the run folder's name on Drive
  (e.g. `'tune_lr1e-3_20261029-101500'`). Training continues from its `last.pt` at the next epoch.


In [ ]:
import yaml
from src.train import train, get_device

EXP = Path('/content/drive/MyDrive/rsvqa_experiments')  # survives disconnects
# Week 3: each run changes one thing from the baseline (hr_pilot_20261028-131854, lr 3e-4).
# No early stopping in tuning runs, so every setting gets its full 10-epoch LR schedule.
NO_EARLY_STOP = {'early_stopping_patience': 10}
WIDE = {'learning_rate': 0.00015, 'd_model': 256, 'num_transformer_layers': 4, 'num_heads': 8,
        'ffn_dim': 512, 'question_embed_dim': 256, 'fusion_hidden_dim': 512}
TUNING = {
    'tune_lr1e-3': {'learning_rate': 0.001},  # done: worse, stopped early at epoch 5
    'tune_lr1.5e-4': {'learning_rate': 0.00015, **NO_EARLY_STOP},  # done: ties with 3e-4
    # larger transformer/fusion (0.9M -> 3.3M params), same CNN; compare with tune_lr1.5e-4
    'tune_wide': {**WIDE, **NO_EARLY_STOP},  # done: unstable, validation collapsed after epoch 3
    'tune_wide_clip': {**WIDE, 'grad_clip_norm': 1.0, **NO_EARLY_STOP},  # done: collapsed at epoch 3 too
    # run-to-run spread: tune_lr1.5e-4 (seed 0) repeated with other seeds
    'tune_lr1.5e-4_seed1': {'learning_rate': 0.00015, 'seed': 1, **NO_EARLY_STOP},
    'tune_lr1.5e-4_seed2': {'learning_rate': 0.00015, 'seed': 2, **NO_EARLY_STOP},
}
RUN = 'tune_lr1.5e-4_seed1'
RESUME = None  # e.g. 'tune_lr1e-3_20261029-101500' to continue a run that was cut off

print('device:', get_device())  # should be cuda
if RESUME:
    best_checkpoint = train({}, resume_dir=EXP / RESUME)
else:
    config = yaml.safe_load(Path('configs/hr_config.yaml').read_text())
    config['experiments_dir'] = str(EXP)
    if RUN:
        config.update(TUNING[RUN])
        config['experiment_name'] = RUN
    print({k: config[k] for k in ('experiment_name', 'learning_rate', 'epochs', 'd_model', 'num_transformer_layers', 'max_images_per_split')})
    best_checkpoint = train(config)
run_dir = best_checkpoint.parent.parent
print('run dir:', run_dir)


## 5b. Optional: diagnose a run whose validation collapsed

Evaluates the run's `last.pt` five ways (val as trained, val in float32, val with dropout on, train data
in eval mode, val after recomputing BatchNorm statistics) to find what separates its good train loss from
its bad validation. About 15 minutes. Set `DIAG` to the run folder's name.


In [ ]:
EXP = Path('/content/drive/MyDrive/rsvqa_experiments')
DIAG = 'tune_wide_YYYYMMDD-HHMMSS'  # the collapsed run's folder name on Drive
!python scripts/diagnose_run.py {EXP / DIAG} configs/hr_config.yaml last.pt 200


## 6. Evaluate on both test sets (`test` and `test_phili`)

Only for runs whose final numbers you want to report; tuning decisions use validation (`log.jsonl`), never test.
If the session reset since step 5, run steps 1-3 again first; this cell then evaluates the latest `hr_pilot` run on Drive.


In [ ]:
EXP = Path('/content/drive/MyDrive/rsvqa_experiments')
if 'run_dir' not in globals():
    run_dir = max(EXP.glob('hr_pilot_*'), key=lambda p: p.stat().st_mtime)
print('evaluating', run_dir)
!python -m src.evaluate {run_dir} configs/hr_config.yaml
!cp reports/baseline_results_{run_dir.name}.md {EXP}/

## Optional: test-set results for the LR `week2_crossattn` run

Needs `rsvqa_lr.zip` in this account's My Drive (uploaded for the earlier LR runs).

In [ ]:
EXP = Path('/content/drive/MyDrive/rsvqa_experiments')
if not Path('data/raw/rsvqa_lr').exists():
    subprocess.run(['unzip', '-q', '-n', '/content/drive/MyDrive/rsvqa_lr.zip', '-d', 'data/raw'], check=True)
lr_run = max(EXP.glob('week2_crossattn_*'), key=lambda p: p.stat().st_mtime)
print('evaluating', lr_run)
!python -m src.evaluate {lr_run}
!cp reports/baseline_results_{lr_run.name}.md {EXP}/